# **Import Libraries**

In [170]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
from scipy.stats import f
plt.style.use('ggplot')
import matplotlib.pyplot as plt 
import seaborn as sns

# **Data Preparation**

In [171]:
data = pd.read_csv('wdi_data.csv')
data['Value'] = pd.to_numeric(data['Value'], errors = 'coerce')
data['Time'] = pd.to_datetime(data['Time'])
data.sort_values(by = ['Country Name', 'Time', 'Series Name'])
data.head(2)

,Country Name,Country Code,Series Name,Series Code,Time,Time Code,Value
0,Bangladesh,BGD,"Agriculture, forestry, and fishing, value adde...",NV.AGR.TOTL.ZS,1970-01-01 00:00:00.000002025,YR2025,11.424022
1,Bangladesh,BGD,"Agriculture, forestry, and fishing, value adde...",NV.AGR.TOTL.ZS,1970-01-01 00:00:00.000002024,YR2024,11.164801


In [172]:
print(f'Number of unique countries: {data['Country Code'].nunique()}')
print(f'Number of lags: {data['Time'].nunique()}')
print(f'Number of Indicators: {data['Series Code'].nunique()}')

Number of unique countries: 9
Number of lags: 50
Number of Indicators: 24


In [173]:
series = pd.DataFrame({'Series' : data['Series Name'].unique()})
series

,Series
0,"Agriculture, forestry, and fishing, value adde..."
1,Exports of goods and services (% of GDP)
2,"Foreign direct investment, net inflows (% of GDP)"
3,GDP (constant 2015 US$)
4,GDP growth (annual %)
5,GDP per capita (constant 2015 US$)
6,GDP per capita growth (annual %)
7,Gross fixed capital formation (% of GDP)
8,Gross fixed capital formation (annual % growth)
9,Gross fixed capital formation (constant 2015 US$)


In [174]:
print('Top countries with the highest maximum foreign investments % of GDP')

(
    data[data['Series Name'] == 'Foreign direct investment, net inflows (% of GDP)']
    .groupby('Country Name')['Value']
    .agg(max_ever_percentage='max', median_percentage = 'median', min_percentage = 'min')
    .sort_values(by='max_ever_percentage', ascending=False)
    .head(5)
)

Top countries with the highest maximum foreign investments % of GDP


,max_ever_percentage,median_percentage,min_percentage
Country Name,,,
Singapore,32.262485,15.268453,3.646085
Viet Nam,11.939483,4.278323,-0.000568
Malaysia,8.760474,3.501092,0.056692
China,6.161583,2.149573,0.000045
Philippines,3.122388,1.307479,-0.287668


In [175]:
print('Top countries with the sorted GPD per capita growth')
data[data['Series Name'] == 'GDP per capita growth (annual %)'].groupby('Country Name')['Value']\
.agg(median_growth_percentage = 'median',
     mean_growth_percentage='mean',
     max_growth_percentage='max').sort_values(by = 'median_growth_percentage', ascending = False)

Top countries with the sorted GPD per capita growth


,median_growth_percentage,mean_growth_percentage,max_growth_percentage
Country Name,,,
China,7.630738,7.722661,13.690607
"Korea, Rep.",5.271189,5.282195,11.806326
Viet Nam,5.237177,4.960347,7.725699
Indonesia,4.174277,3.541409,7.286559
Singapore,4.128894,4.074477,14.831461
Malaysia,3.761559,3.378917,8.922229
Bangladesh,3.283302,3.416922,6.972644
Philippines,2.764092,1.970102,6.765954
Japan,1.825062,1.730927,6.341600


In [176]:
data = data.pivot(index = ['Country Name', 'Time'], columns = 'Series Name', values = 'Value').reset_index().copy()
data['Time'] = pd.to_numeric(data['Time'])
data.head()

Series Name,Country Name,Time,"Agriculture, forestry, and fishing, value added (% of GDP)",Exports of goods and services (% of GDP),"Foreign direct investment, net inflows (% of GDP)",GDP (constant 2015 US$),GDP growth (annual %),GDP per capita (constant 2015 US$),GDP per capita growth (annual %),Gross fixed capital formation (% of GDP),...,"Manufacturing, value added (annual % growth)","Manufacturing, value added (constant 2015 US$)",Population growth (annual %),"Population, female","Population, female (% of total population)","Population, male","Population, male (% of total population)","Population, total","Services, value added (% of GDP)",Trade (% of GDP)
0,Bangladesh,1976,51.914457,4.746082,0.053573,3.070513e+10,5.661361,384.509552,3.013762,NaN,...,-0.690863,2.446962e+09,2.537669,38756724.0,48.533679,41098594.0,51.466321,79855318.0,33.636340,22.382275
1,Bangladesh,1977,48.898057,7.040752,0.072323,3.152590e+10,2.673056,385.085520,0.149793,NaN,...,11.659954,2.732277e+09,2.488273,39758252.0,48.564286,42109012.0,51.435714,81867264.0,34.372869,19.422198
2,Bangladesh,1978,54.250308,5.557663,0.057974,3.375599e+10,7.073838,402.385596,4.492528,NaN,...,1.177488,2.764449e+09,2.440310,40766279.0,48.595121,43123376.0,51.404879,83889655.0,31.439979,21.038515
3,Bangladesh,1979,52.452012,6.109219,-0.051460,3.537683e+10,4.801635,411.669267,2.307158,NaN,...,11.190006,3.073791e+09,2.408973,41786802.0,48.626016,44148270.0,51.373984,85935072.0,31.861533,21.907064
4,Bangladesh,1980,32.823892,5.494284,0.046918,3.566661e+10,0.819142,405.226770,-1.564969,14.439391,...,2.325027,3.145257e+09,2.393149,42826188.0,48.657037,45190244.0,51.342963,88016432.0,44.555666,23.377231


**Chow's Test**

If there is a split in between the sample size for any given cut off points by the $N = N_1 + N_2$

The Hypothesis:

$Null \ Hypotehsis \ H_0:$ 

$\beta_{1, 0} = \beta_{2, 0},\ \beta_{1, 1} = \beta_{2, 1},\ \dots,\ \beta_{1, k} = \beta_{2, k}$

*(A single pooled regression line represents the entire dataset just as effectively as two separate regressions.)*

Alternative Hypothesis ($H_1$): At least one parameter differs between the two groups ($\beta_{1,j} \neq \beta_{2,j}$ for some $j$), indicating a structural break.

In [177]:
def chow_test(data, time_col, break_year, y_col, x_cols):
    df_1 = data[data[time_col] < break_year]
    df_2 = data[data[time_col] >= break_year]
    
    y_p = data[y_col]
    X_p = sm.add_constant(data[x_cols])
    model_p = sm.OLS(y_p, X_p).fit()
    rss_p = model_p.ssr
    coeffs = model_p.params[x_cols]
    
    y_1 = df_1[y_col]
    X_1 = sm.add_constant(df_1[x_cols])
    rss_1 = sm.OLS(y_1, X_1).fit().ssr
    
    y_2 = df_2[y_col]
    X_2 = sm.add_constant(df_2[x_cols])
    rss_2 = sm.OLS(y_2, X_2).fit().ssr
    
    k = X_p.shape[1]
    n_1 = len(df_1)
    n_2 = len(df_2)
    
    numerator = (rss_p - (rss_1 + rss_2)) / k
    denominator = (rss_1 + rss_2) / (n_1 + n_2 - 2 * k)
    
    f_stat = numerator / denominator
    p_val = 1 - f.cdf(f_stat, k, n_1 + n_2 - 2 * k)
    
    return f_stat, p_val, coeffs


In [178]:

y_variable = 'GDP growth (annual %)'
x_variables = ['Gross fixed capital formation (% of GDP)', 'Exports of goods and services (% of GDP)', 'Manufacturing, value added (% of GDP)', 'Trade (% of GDP)']

def condition(country:str, cutoff:int, x = x_variables, y = y_variable, display:bool=True, alpha:float=0.05, coefficients:bool = False):
    country_df = data[data['Country Name'] == country].sort_values('Time').dropna(
    subset=x + [y]
    )
    f_statistic, p_value, coeff = chow_test(country_df, 'Time', cutoff, y, x)
    
    decision = f'Decision: {'There was an effect' if p_value<alpha else 'There was no effect'}'
    if display == True:
        print(f"Chow's Test Statistics Results for {cutoff} F-statistic: {f_statistic:.4f}, p-value: {p_value:.4e}")
        plt.figure(figsize = (20, 10))
        sns.lineplot(data = country_df,  x = 'Time', y = y, palette = 'magma', linewidth = 2)
        plt.text(cutoff, country_df[y].quantile(0.75), decision+' on '+f'{cutoff}', fontsize=14, fontweight='bold', color='red' if decision == 'Decision: There was an effect' else 'green', bbox=dict(facecolor='white', alpha=0.8, edgecolor='red'))
        plt.axvline(x=cutoff, color='red', linestyle='--')
        plt.title(f'GDP growth (annual %) for {country}')
        plt.show()
    if coefficients == True:
        print('Here are the coefficient results for Chow Test:')
        print(coeff)

    return decision


### Observing Country-wise GDP Growth Annually

In [179]:

effects = {
    'Country Name': [],
    'Years': []
}
for c in data['Country Name'].unique():
    country_check = data[data['Country Name'] == c].dropna(subset=x_variables + [y_variable])
    if len(country_check) <= 2 * (len(x_variables) + 1):
        continue
    effective_times = []
    for i in range(1995, 2021):
        if len(country_check[country_check['Time'] < i]) <= len(x_variables) + 1 or len(country_check[country_check['Time'] >= i]) <= len(x_variables) + 1:
            continue
        decision = condition(c, i, display = False, alpha = 0.05)
        if decision == 'Decision: There was an effect':
            effective_times.append(i)
    effects['Years'].append(effective_times)
    effects['Country Name'].append(c)

pd.DataFrame(effects)

,Country Name,Years
0,Bangladesh,"[2016, 2017, 2020]"
1,China,[2019]
2,Indonesia,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200..."
3,Japan,[2009]
4,"Korea, Rep.","[1995, 1996, 1997, 1998, 2000, 2001, 2002, 200..."
5,Malaysia,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200..."
6,Philippines,"[2016, 2017, 2018, 2019, 2020]"
7,Singapore,[2010]
8,Viet Nam,[2020]


In [191]:
for i in range(2010, 2022):
    decision = condition('Viet Nam', i, display = False, alpha = 0.05, coefficients = False)
    print(i, ':' , decision if decision == 'Decision: There was an effect' else 'None')

2010 : None
2011 : None
2012 : None
2013 : None
2014 : None
2015 : None
2016 : None
2017 : None
2018 : None
2019 : None
2020 : Decision: There was an effect
2021 : None
